In [2]:
#dependencias
import geopandas as gpd
import matplotlib.pyplot as plt
import re
import pandas as pd
import sys
from pathlib import Path

In [ ]:
#Configuración de la ruta del archivo
# .shp, .dbf, .shx y .prj en la misma carpeta

#ubuntu
#ruta = '/home/ubuntu/projects/analisis_espacial_marginalNoise/'

#windows
ruta = 'E:/PROYECTOS/analisis_espacial_marginalNoise/'
%store ruta

Path(ruta) / 'data' / 'RedVial' / 'RedVial.shp'

archivo_shapefile = Path(ruta) / 'data' / 'RedVial' / 'RedVial.shp'

try:
    #Cargar la red vial
    red_vial = gpd.read_file(archivo_shapefile)
    
    print(f"Sistema de Coordenadas (CRS): {red_vial.crs}")
    print(f"Cantidad de tramos: {len(red_vial)}")
    print("\nPrimeras filas de datos (Atributos):")
    display(red_vial.head())

    #Visualización geometrica básica
    fig, ax = plt.subplots(figsize=(12, 12))
    red_vial.plot(ax=ax, color='blue', linewidth=0.5, alpha=0.7)
    ax.set_title("Visualización de la Red Vial", fontsize=16)
    ax.set_xlabel("Coordenada X")
    ax.set_ylabel("Coordenada Y")
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except Exception as e:
    print(f"Error al leer el archivo: {e}")
    print("Verifica que la ruta sea correcta y que tengas instaladas las dependencias.")

Stored 'ruta' (str)
Error al leer el archivo: '\home\ubuntu\projects\analisis_espacial_marginalNoise\data\RedVial\RedVial.shp' does not exist in the file system, and is not recognized as a supported dataset name.
Verifica que la ruta sea correcta y que tengas instaladas las dependencias.


In [ ]:
# Transforma el string '"key"=>"value"' en un diccionario de Python
def parse_osm_tags(tag_str):
    if not isinstance(tag_str, str):
        return {}
    pattern = r'"([^"]+)"=>"([^"]+)"'
    matches = re.findall(pattern, tag_str)
    return dict(matches)

def estimar_velocidad(row):
    if pd.notna(row.get('maxspeed')):
        speed = re.findall(r'\d+', str(row['maxspeed']))
        return int(speed[0]) if speed else 40
    
    tipo = row['highway']
    if tipo == 'motorway': return 100
    elif tipo == 'primary': return 60
    elif tipo == 'secondary': return 50
    elif tipo == 'tertiary': return 40
    else: return 30

#PROCESAMIENTO
print("Extrayendo velocidad y carriles de 'other_tags'...")
tags_parsed = red_vial['other_tags'].apply(parse_osm_tags)
df_tags = pd.DataFrame(tags_parsed.tolist())
red_vial = pd.concat([red_vial, df_tags], axis=1)

red_vial = red_vial.loc[:, ~red_vial.columns.duplicated()].copy()
red_vial['velocidad_vf'] = red_vial.apply(estimar_velocidad, axis=1)

if red_vial.crs.is_geographic:
    print("Reproyectando mapa a UTM Zona 19S (Metros)...")
    red_vial = red_vial.to_crs(epsg=32719)


#VISUALIZACION
print("\nMapa de Velocidades Máximas (Input para Greenshields)")
fig, ax = plt.subplots(figsize=(12, 10))

red_vial.plot(column='velocidad_vf', 
            ax=ax, 
            legend=True,
            cmap='plasma',
            legend_kwds={'label': "Velocidad de Flujo Libre ($v_f$) [km/h]"},
            linewidth=1)

ax.set_title("Red Vial: Velocidades de Flujo Libre ($v_f$)", fontsize=15)
ax.set_xlabel("Metros (Este)")
ax.set_ylabel("Metros (Norte)")
plt.show()

# Verificación de datos
print("\nPrimeras filas con datos extraídos:")
cols_interes = ['name', 'highway', 'maxspeed', 'lanes', 'velocidad_vf']
cols_mostrar = [c for c in cols_interes if c in red_vial.columns]
display(red_vial[cols_mostrar].head())

%store red_vial

Extrayendo velocidad y carriles de 'other_tags'...


NameError: name 'red_vial' is not defined

In [ ]:
# mapa interactivo para mayor resolucion
paleta = 'plasma_r'  # Puedes cambiar a 'viridis', 'magma', 'coolwarm', etc.

mapa_interactivo = red_vial.explore(
    column='velocidad_vf', # Columna para colorear
    cmap=paleta,
    tiles='CartoDB positron',   # Fondo del mapa (limpio y gris para resaltar datos)
    style_kwds={
        'weight': 3, # Grosor de la línea
        'opacity': 0.8 # Transparencia
    },
    tooltip=['name', 'highway', 'maxspeed', 'lanes', 'velocidad_vf'], # Datos al pasar el mouse
    popup=True, # Al hacer clic muestra todos los datos
    legend=True # Mostrar leyenda de colores
)

mapa_interactivo.save(Path( ruta ) / 'data' / 'data_generada' / 'mapa_velocidades.html')
print("\nMapa velocidades guardado")

#guardar red vial procesada
red_vial.to_file(Path(ruta) / 'data' / 'data_generada' / 'red_vial_procesada.gpkg', driver="GPKG")
print("Archivo 'red_vial_procesada.gpkg' guardado")


Mapa velocidades guardado
Archivo 'red_vial_procesada.gpkg' guardado
